In [1]:
import sys
import os
from pprint import pprint

sys.path.append(os.path.dirname(os.getcwd()))
# from src.core import * # type: ignore
from src.hamiltonians import *
from src.lattices import *
from src.propagators import *
from src.states import *
from src.walkers import *

In [2]:
Nsites = 100
Lx=10
Ly=10
Nup = Lx * Ly // 2
Ndown = Lx * Ly // 2
# Nup = Lx * Ly // (2 * 8) 
# Ndown = Lx * Ly // (2 * 8)
t = 1.0
U = 0.0
dtau = 0.05
n_steps = 5000

In [3]:
# lattice = Chain(
#     n_sites=Nsites,
#     pbc=False
# )
lattice = Square(
    Lx=Lx,
    Ly=Lx,
    pbc=True
)

system = HubbardSystem(
    lattice=lattice,
    t=t,
    U=U
)

K = system.h_kin
# K

In [4]:
trial = SlaterDeterminantTwoSpinState(
    hamiltonian=system,
    n_electrons_up=Nup,
    n_electrons_down=Ndown
)

trial.initialize("non-interacting")

evals = system.get_non_interacting_evals_evecs(
    return_evals=True,
    return_evecs=False
)

E_exact = (
    np.sum(evals[:Nup])
    +
    np.sum(evals[:Ndown])
)

print()
print("Exact U=0 ground-state energy")
print(E_exact)
print(evals)


Exact U=0 ground-state energy
-159.55417527999333
[-4.00000000e+00 -3.61803399e+00 -3.61803399e+00 -3.61803399e+00
 -3.61803399e+00 -3.23606798e+00 -3.23606798e+00 -3.23606798e+00
 -3.23606798e+00 -2.61803399e+00 -2.61803399e+00 -2.61803399e+00
 -2.61803399e+00 -2.23606798e+00 -2.23606798e+00 -2.23606798e+00
 -2.23606798e+00 -2.23606798e+00 -2.23606798e+00 -2.23606798e+00
 -2.23606798e+00 -1.38196601e+00 -1.38196601e+00 -1.38196601e+00
 -1.38196601e+00 -1.23606798e+00 -1.23606798e+00 -1.23606798e+00
 -1.23606798e+00 -1.00000000e+00 -1.00000000e+00 -1.00000000e+00
 -1.00000000e+00 -1.00000000e+00 -1.00000000e+00 -1.00000000e+00
 -1.00000000e+00 -3.81966011e-01 -3.81966011e-01 -3.81966011e-01
 -3.81966011e-01 -1.50938161e-15 -1.46392390e-15 -9.12461216e-16
 -7.07509187e-16 -6.33052328e-16 -6.15199999e-16 -5.22591517e-16
 -1.95761279e-16 -1.91218232e-16 -1.99264826e-17  6.01194387e-17
  1.15174184e-16  4.15795614e-16  5.75044520e-16  7.66571080e-16
  8.29960848e-16  1.10805728e-15  1.367

In [5]:
state = SlaterDeterminantTwoSpinState(
    hamiltonian=system,
    n_electrons_up=Nup,
    n_electrons_down=Ndown
)

state.initialize("random")

walker = Walker(state)

prop = HubbardPropagator(
    K=K,
    U=U,
    dtau=dtau
)

print()
print("gamma =", prop.gamma)
print()

for step in range(n_steps):

    prop.propagate(walker)

    walker.orthogonalize()

    if step % 20 == 0:

        overlap = trial.overlap_calculation_logdet(
            walker.state
        )

        print(
            f"step = {step:4d}"
            f"   overlap = {abs(overlap):.12f}"
        )


final_overlap = trial.overlap_calculation_logdet(
    walker.state
)

print()
print("Final overlap:")
print(abs(final_overlap))

error_up = np.linalg.norm(
    walker.state.phi_up @ walker.state.phi_up.conj().T
    -
    trial.phi_up @ trial.phi_up.conj().T
)

error_down = np.linalg.norm(
    walker.state.phi_down @ walker.state.phi_down.conj().T
    -
    trial.phi_down @ trial.phi_down.conj().T
)

print()
print("Projector error (up)   =", error_up)
print("Projector error (down) =", error_down)



gamma = 0.0

step =    0   overlap = 0.000000000000
step =   20   overlap = 0.000000006661
step =   40   overlap = 0.000000465707
step =   60   overlap = 0.000001299976
step =   80   overlap = 0.000001892448
step =  100   overlap = 0.000002229724
step =  120   overlap = 0.000002404073
step =  140   overlap = 0.000002489647
step =  160   overlap = 0.000002530538
step =  180   overlap = 0.000002549819
step =  200   overlap = 0.000002558853
step =  220   overlap = 0.000002563073
step =  240   overlap = 0.000002565041
step =  260   overlap = 0.000002565959
step =  280   overlap = 0.000002566386
step =  300   overlap = 0.000002566585
step =  320   overlap = 0.000002566678
step =  340   overlap = 0.000002566721
step =  360   overlap = 0.000002566741
step =  380   overlap = 0.000002566751
step =  400   overlap = 0.000002566755
step =  420   overlap = 0.000002566757
step =  440   overlap = 0.000002566758
step =  460   overlap = 0.000002566758
step =  480   overlap = 0.000002566759
step =  500

In [6]:
def energy(phi, K):
    return np.trace(phi.conj().T @ K @ phi).real

E_proj = (energy(walker.state.phi_up, K) + energy(walker.state.phi_down, K))

print(E_proj)
print(E_exact)
print(np.abs((E_proj - E_proj) / E_exact) * 100)

-159.55417527999327
-159.55417527999333
0.0
